In [ ]:
%pip -q install depthai==3.11.0

In [ ]:
import cv2
import depthai as dai
import numpy as np
import json
import math
import random
import re
import shutil
import zipfile
from pathlib import Path
from IPython.display import display
from google.colab import files
import pandas as pd
import matplotlib.pyplot as plt

IMAGE_WIDTH = 640
IMAGE_HEIGHT = 400
IMAGE_SIZE = (IMAGE_WIDTH, IMAGE_HEIGHT)
CAM_LEFT_ID = 1
CAM_RIGHT_ID = 2
CAM_LEFT = dai.CameraBoardSocket.CAM_B
CAM_RIGHT = dai.CameraBoardSocket.CAM_C
# These dimensions belong ONLY to the built-in synthetic test.
TEST_PATTERN_SIZE = (9, 6)
TEST_SQUARE_SIZE_MM = 25.0
PATTERN_SIZE = TEST_PATTERN_SIZE
SQUARE_SIZE_MM = TEST_SQUARE_SIZE_MM
MIN_PAIRS = 20
MIN_TRAIN_PAIRS = 15
MAX_MONO_RMS_PX = 1.5
MAX_STEREO_RMS_PX = 1.5
MAX_VALIDATION_REPROJECTION_PX = 1.5
MAX_EPIPOLAR_RMS_PX = 0.70
MAX_BASELINE_CHANGE_PERCENT = 15.0
RANDOM_SEED = 42
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
WORK_DIR = Path("/content/oak_stereo_calibration")
OUTPUT_PATH = Path("/content/calibration_data.json")

print("OpenCV:", cv2.__version__)
print("DepthAI:", getattr(dai, "__version__", "installed"))
print("Camera pair: CAM_B -> CAM_C | 640 x 400 | GRAY8")

In [ ]:
def ask_choice(prompt, choices):
    lookup = {str(value).lower(): value for value in choices}
    while True:
        answer = input(f"{prompt} ({'/'.join(choices)}): ").strip().lower()
        if answer in lookup:
            return lookup[answer]
        print("Choose:", ", ".join(choices))


def ask_positive_int(prompt, minimum=2):
    while True:
        answer = input(prompt).strip()
        try:
            value = int(answer)
            if value >= minimum:
                return value
        except ValueError:
            pass
        print(f"Enter a whole number greater than or equal to {minimum}.")


def ask_positive_float(prompt):
    while True:
        answer = input(prompt).strip()
        try:
            value = float(answer)
            if math.isfinite(value) and value > 0:
                return value
        except ValueError:
            pass
        print("Enter a valid number greater than zero.")


def extract_zip_safely(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        for item in archive.infolist():
            target = (destination / item.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError("The ZIP contains an unsafe file path.")
        archive.extractall(destination)
    return destination


def make_builtin_test(destination):
    destination = Path(destination)
    left_dir = destination / "CAM_B"
    right_dir = destination / "CAM_C"
    left_dir.mkdir(parents=True, exist_ok=True)
    right_dir.mkdir(parents=True, exist_ok=True)

    K_left = np.array([
        [430.626953125, 0.0, 320.33416748046875],
        [0.0, 430.3931884765625, 201.5816192626953],
        [0.0, 0.0, 1.0]
    ], dtype=np.float64)
    K_right = np.array([
        [430.0867614746094, 0.0, 318.8838806152344],
        [0.0, 429.73675537109375, 204.40399169921875],
        [0.0, 0.0, 1.0]
    ], dtype=np.float64)
    D_left = np.array([-0.035, 0.008, 0.0003, -0.0002, 0.001], dtype=np.float64)
    D_right = np.array([-0.032, 0.006, 0.0002, -0.0002, 0.001], dtype=np.float64)
    R_lr = np.array([
        [0.9998782277107239, 0.010038220323622227, 0.011950093321502209],
        [-0.010067159633636, 0.9999465346336365, 0.0023639767896384],
        [-0.01192572433501482, -0.0024839923717081547, 0.9999257922172546]
    ], dtype=np.float64)
    T_lr_mm = np.array([-75.20867, 0.562431, 0.602984], dtype=np.float64).reshape(3, 1)

    cols, rows = PATTERN_SIZE
    square = SQUARE_SIZE_MM
    center = np.array([(cols - 1) * square / 2.0, (rows - 1) * square / 2.0, 0.0])
    rng = np.random.default_rng(20261030)
    made = 0
    attempts = 0

    def project_square(image, R_cam, t_cam, K, D, c, r, color):
        points = np.array([
            [(c - 1) * square, (r - 1) * square, 0.0],
            [c * square, (r - 1) * square, 0.0],
            [c * square, r * square, 0.0],
            [(c - 1) * square, r * square, 0.0]
        ], dtype=np.float32)
        rvec, _ = cv2.Rodrigues(R_cam)
        projected, _ = cv2.projectPoints(points, rvec, t_cam, K, D)
        polygon = np.round(projected.reshape(-1, 2)).astype(np.int32)
        cv2.fillConvexPoly(image, polygon, int(color), lineType=cv2.LINE_AA)
        return polygon

    while made < 36 and attempts < 250:
        attempts += 1
        rotation_vector = np.deg2rad(rng.uniform([-22, -25, -32], [22, 25, 32]))
        R_board, _ = cv2.Rodrigues(rotation_vector)
        z = float(rng.uniform(560.0, 820.0))
        center_t = np.array([
            rng.uniform(-145.0, 145.0),
            rng.uniform(-85.0, 85.0),
            z
        ], dtype=np.float64).reshape(3, 1)
        t_left = center_t - R_board @ center.reshape(3, 1)
        R_right = R_lr @ R_board
        t_right = R_lr @ t_left + T_lr_mm

        left_image = np.full((IMAGE_HEIGHT, IMAGE_WIDTH), 180, dtype=np.uint8)
        right_image = np.full((IMAGE_HEIGHT, IMAGE_WIDTH), 180, dtype=np.uint8)
        left_polygons, right_polygons = [], []
        for row in range(rows + 1):
            for col in range(cols + 1):
                shade = 245 if (row + col) % 2 == 0 else 15
                left_polygons.append(project_square(left_image, R_board, t_left, K_left, D_left, col, row, shade))
                right_polygons.append(project_square(right_image, R_right, t_right, K_right, D_right, col, row, shade))

        all_points = np.vstack(left_polygons + right_polygons)
        if (all_points[:, 0].min() < 5 or all_points[:, 0].max() >= IMAGE_WIDTH - 5
                or all_points[:, 1].min() < 5 or all_points[:, 1].max() >= IMAGE_HEIGHT - 5):
            continue

        gain = float(rng.uniform(0.92, 1.08))
        bias = float(rng.uniform(-5.0, 5.0))
        for image in (left_image, right_image):
            adjusted = np.clip(image.astype(np.float32) * gain + bias, 0, 255).astype(np.uint8)
            noise = rng.normal(0, 0.6, adjusted.shape).astype(np.float32)
            adjusted = np.clip(adjusted.astype(np.float32) + noise, 0, 255).astype(np.uint8)
            if rng.random() < 0.25:
                adjusted = cv2.GaussianBlur(adjusted, (3, 3), 0.35)
            if image is left_image:
                left_image = adjusted
            else:
                right_image = adjusted

        flags = cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE
        found_left, _ = cv2.findChessboardCorners(left_image, PATTERN_SIZE, flags)
        found_right, _ = cv2.findChessboardCorners(right_image, PATTERN_SIZE, flags)
        if not (found_left and found_right):
            continue

        made += 1
        cv2.imwrite(str(left_dir / f"{made:03d}.png"), left_image)
        cv2.imwrite(str(right_dir / f"{made:03d}.png"), right_image)

    if made < MIN_PAIRS:
        raise RuntimeError(f"Built-in test generation created only {made} usable pairs.")

    D14_left = np.pad(D_left, (0, 9)).tolist()
    D14_right = np.pad(D_right, (0, 9)).tolist()
    T_cm = (T_lr_mm.reshape(3) / 10.0)
    R_rl = R_lr.T
    T_rl_cm = (-R_rl @ T_lr_mm.reshape(3) / 10.0)

    def camera_record(K, D, extrinsics):
        return {
            "cameraType": 0,
            "distortionCoeff": D,
            "extrinsics": extrinsics,
            "height": IMAGE_HEIGHT,
            "intrinsicMatrix": K.tolist(),
            "lensPosition": 0,
            "specHfovDeg": 71.86,
            "width": IMAGE_WIDTH
        }

    left_extrinsics = {
        "lengthUnit": 1,
        "rotationMatrix": R_lr.tolist(),
        "specTranslation": {"x": -7.5, "y": 0.0, "z": 0.0},
        "toCameraSocket": CAM_RIGHT_ID,
        "toDeviceId": "",
        "translation": {"x": float(T_cm[0]), "y": float(T_cm[1]), "z": float(T_cm[2])}
    }
    right_extrinsics = {
        "lengthUnit": 1,
        "rotationMatrix": R_rl.tolist(),
        "specTranslation": {"x": 7.5, "y": 0.0, "z": 0.0},
        "toCameraSocket": CAM_LEFT_ID,
        "toDeviceId": "",
        "translation": {"x": float(T_rl_cm[0]), "y": float(T_rl_cm[1]), "z": float(T_rl_cm[2])}
    }
    reference = {
        "batchName": "Synthetic smoke test only",
        "batchTime": 0,
        "boardConf": "",
        "boardCustom": "",
        "boardName": "BW1098OBC",
        "boardOptions": 0,
        "boardRev": "R0M0E0",
        "cameraData": [
            [CAM_LEFT_ID, camera_record(K_left, D14_left, left_extrinsics)],
            [CAM_RIGHT_ID, camera_record(K_right, D14_right, right_extrinsics)]
        ],
        "deviceName": "Synthetic test data only",
        "hardwareConf": "",
        "housingExtrinsics": {
            "lengthUnit": 1, "rotationMatrix": [],
            "specTranslation": {"x": 0.0, "y": 0.0, "z": 0.0},
            "toCameraSocket": -1, "toDeviceId": "",
            "translation": {"x": 0.0, "y": 0.0, "z": 0.0}
        },
        "productName": "",
        "stereoEnableDistortionCorrection": False,
        "stereoRectificationData": {
            "leftCameraSocket": CAM_LEFT_ID,
            "rectifiedRotationLeft": np.eye(3).tolist(),
            "rectifiedRotationRight": np.eye(3).tolist(),
            "rightCameraSocket": CAM_RIGHT_ID
        },
        "stereoUseSpecTranslation": False,
        "version": 6,
        "verticalCameraSocket": -1
    }
    reference_path = destination / "factory_calibration.json"
    with reference_path.open("w", encoding="utf-8") as handle:
        json.dump(reference, handle, indent=2)
    (destination / "TEST_DATA_INFO.txt").write_text(
        "SYNTHETIC TEST DATA ONLY. Not real OAK captures. Do not use the output on a real ROV.\n",
        encoding="utf-8"
    )
    return destination


MODE = ask_choice("Data source", ["Upload My Dataset", "Run Built-in Test"])
ENVIRONMENT = ask_choice("Capture environment", ["Air", "Underwater"])
IS_BUILTIN_TEST = MODE == "Run Built-in Test"

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()

if IS_BUILTIN_TEST:
    DATA_DIR = make_builtin_test(WORK_DIR / "dataset")
    print("Built-in synthetic test created. It checks the notebook workflow only.")
else:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        use_test = ask_choice("No ZIP was uploaded. Run the built-in test instead?", ["Yes", "No"])
        if use_test == "Yes":
            IS_BUILTIN_TEST = True
            PATTERN_SIZE = TEST_PATTERN_SIZE
            SQUARE_SIZE_MM = TEST_SQUARE_SIZE_MM
            DATA_DIR = make_builtin_test(WORK_DIR / "dataset")
            print("Built-in synthetic test created. It does not represent your camera.")
        else:
            raise RuntimeError("Upload exactly one ZIP file and rerun this cell.")
    else:
        ZIP_PATH = Path("/content") / zip_names[0]
        DATA_DIR = extract_zip_safely(ZIP_PATH, WORK_DIR / "dataset")
        print("Dataset:", ZIP_PATH.name)

        # Ask for the REAL calibration target dimensions only for uploaded data.
        CORNER_COLS = ask_positive_int("Chessboard INNER corners — number of columns: ")
        CORNER_ROWS = ask_positive_int("Chessboard INNER corners — number of rows: ")
        SQUARE_SIZE_MM = ask_positive_float("Actual side length of ONE square (millimeters): ")
        PATTERN_SIZE = (CORNER_COLS, CORNER_ROWS)

print("Environment:", ENVIRONMENT)
print("Pattern:", PATTERN_SIZE, "inner corners | square:", SQUARE_SIZE_MM, "mm")

In [ ]:
def find_reference_json(folder):
    candidates = [path for path in Path(folder).rglob("*.json") if path.is_file()]
    preferred = ["factory_calibration.json", "calibration.json", "user_calibration.json"]
    for filename in preferred:
        matches = [path for path in candidates if path.name.lower() == filename]
        if matches:
            return sorted(matches)[0]
    if len(candidates) == 1:
        return candidates[0]
    raise FileNotFoundError(
        "The ZIP must contain one OAK calibration JSON, preferably named factory_calibration.json."
    )


def collect_pairs(folder):
    left_dirs = {"left", "cam_b", "camera_b", "mono_left", "left_images"}
    right_dirs = {"right", "cam_c", "camera_c", "mono_right", "right_images"}
    left_suffix = re.compile(r"(?:[_ .-](?:left|cam[_-]?b|camera[_-]?b|mono[_-]?left))$", re.I)
    right_suffix = re.compile(r"(?:[_ .-](?:right|cam[_-]?c|camera[_-]?c|mono[_-]?right))$", re.I)
    left_map, right_map = {}, {}

    for path in Path(folder).rglob("*"):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        if "__macosx" in str(path).lower() or path.name.startswith("."):
            continue
        parent_labels = {part.lower() for part in path.relative_to(folder).parts[:-1]}
        stem = path.stem
        left_match = left_suffix.search(stem)
        right_match = right_suffix.search(stem)
        if parent_labels & left_dirs:
            key = stem[:left_match.start()] if left_match else stem
            left_map[key.lower()] = path
        elif parent_labels & right_dirs:
            key = stem[:right_match.start()] if right_match else stem
            right_map[key.lower()] = path
        elif left_match:
            left_map[stem[:left_match.start()].lower()] = path
        elif right_match:
            right_map[stem[:right_match.start()].lower()] = path

    common = sorted(set(left_map) & set(right_map))
    if not common:
        raise ValueError(
            "No synchronized stereo pairs found. Use CAM_B/ and CAM_C/ folders, "
            "Left/ and Right/ folders, or matching _left/_right filenames."
        )
    return [(key, left_map[key], right_map[key]) for key in common]


def camera_record(calibration_json, socket_id):
    for item in calibration_json.get("cameraData", []):
        if isinstance(item, (list, tuple)) and len(item) == 2 and int(item[0]) == socket_id:
            return item[1]
    raise ValueError(f"Calibration JSON has no camera record for socket {socket_id}.")


def scaled_intrinsics(record):
    matrix = np.asarray(record.get("intrinsicMatrix"), dtype=np.float64)
    source_width = float(record.get("width", 0))
    source_height = float(record.get("height", 0))
    if matrix.shape != (3, 3) or source_width <= 0 or source_height <= 0:
        raise ValueError("Reference camera intrinsics are incomplete.")
    matrix[0, :] *= IMAGE_WIDTH / source_width
    matrix[1, :] *= IMAGE_HEIGHT / source_height
    return matrix


def reference_stereo_extrinsics(reference_json):
    left = camera_record(reference_json, CAM_LEFT_ID)
    right = camera_record(reference_json, CAM_RIGHT_ID)
    ext = left.get("extrinsics", {})
    if int(ext.get("toCameraSocket", -1)) == CAM_RIGHT_ID:
        rotation = np.asarray(ext.get("rotationMatrix"), dtype=np.float64)
        translation = np.array(
            [ext.get("translation", {}).get(axis, 0.0) for axis in ("x", "y", "z")],
            dtype=np.float64
        ).reshape(3, 1)
    else:
        ext = right.get("extrinsics", {})
        if int(ext.get("toCameraSocket", -1)) != CAM_LEFT_ID:
            raise ValueError("Reference JSON must contain CAM_B-to-CAM_C extrinsics.")
        rotation_reverse = np.asarray(ext.get("rotationMatrix"), dtype=np.float64)
        translation_reverse = np.array(
            [ext.get("translation", {}).get(axis, 0.0) for axis in ("x", "y", "z")],
            dtype=np.float64
        ).reshape(3, 1)
        rotation = rotation_reverse.T
        translation = -rotation @ translation_reverse
    if rotation.shape != (3, 3) or not np.all(np.isfinite(rotation)) or not np.all(np.isfinite(translation)):
        raise ValueError("Reference stereo extrinsics are invalid.")
    return rotation, translation, left, right


def skew_symmetric(vector):
    x, y, z = np.asarray(vector, dtype=np.float64).reshape(3)
    return np.array([[0.0, -z, y], [z, 0.0, -x], [-y, x, 0.0]], dtype=np.float64)


def detect_board(gray):
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE
    found, corners = cv2.findChessboardCorners(gray, PATTERN_SIZE, flags)
    if found and corners is not None:
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 1e-3)
        return cv2.cornerSubPix(gray, corners, (7, 7), (-1, -1), criteria)
    if hasattr(cv2, "findChessboardCornersSB"):
        sb_flags = cv2.CALIB_CB_NORMALIZE_IMAGE | getattr(cv2, "CALIB_CB_ACCURACY", 0)
        found, corners = cv2.findChessboardCornersSB(gray, PATTERN_SIZE, flags=sb_flags)
        if found and corners is not None:
            return corners.astype(np.float32).reshape(-1, 1, 2)
    return None


def orientation_candidates(pattern_size):
    cols, rows = pattern_size
    grid = np.arange(cols * rows).reshape(rows, cols)
    return [
        grid.reshape(-1),
        grid[:, ::-1].reshape(-1),
        grid[::-1, :].reshape(-1),
        grid[::-1, ::-1].reshape(-1)
    ]


def sampson_median(points_left, points_right, essential):
    p1 = np.asarray(points_left, dtype=np.float64).reshape(-1, 2)
    p2 = np.asarray(points_right, dtype=np.float64).reshape(-1, 2)
    x1 = np.column_stack([p1, np.ones(len(p1))])
    x2 = np.column_stack([p2, np.ones(len(p2))])
    e_x1 = (essential @ x1.T).T
    et_x2 = (essential.T @ x2.T).T
    numerator = np.sum(x2 * e_x1, axis=1) ** 2
    denominator = e_x1[:, 0] ** 2 + e_x1[:, 1] ** 2 + et_x2[:, 0] ** 2 + et_x2[:, 1] ** 2
    distances = np.sqrt(numerator / np.maximum(denominator, 1e-15))
    return float(np.median(distances))


def align_right_corner_order(corners_left, corners_right, K_left_ref, D_left_ref,
                             K_right_ref, D_right_ref, R_ref, T_ref):
    undistorted_left = cv2.undistortPoints(corners_left, K_left_ref, D_left_ref).reshape(-1, 2)
    undistorted_right = cv2.undistortPoints(corners_right, K_right_ref, D_right_ref).reshape(-1, 2)
    essential = skew_symmetric(T_ref) @ R_ref
    scores = []
    candidates = orientation_candidates(PATTERN_SIZE)
    for indices in candidates:
        score = sampson_median(undistorted_left, undistorted_right[indices], essential)
        scores.append(score)
    best_index = int(np.argmin(scores))
    ordered_right = corners_right.reshape(-1, 2)[candidates[best_index]].reshape(-1, 1, 2)
    sorted_scores = sorted(scores)
    margin = sorted_scores[1] / max(sorted_scores[0], 1e-12)
    return ordered_right.astype(np.float32), best_index, scores, margin

In [ ]:
REFERENCE_PATH = find_reference_json(DATA_DIR)
with REFERENCE_PATH.open("r", encoding="utf-8") as handle:
    REFERENCE_JSON = json.load(handle)

PAIR_FILES = collect_pairs(DATA_DIR)
if len(PAIR_FILES) < MIN_PAIRS:
    raise RuntimeError(f"Only {len(PAIR_FILES)} image pairs were found. At least {MIN_PAIRS} are required.")

left_record = camera_record(REFERENCE_JSON, CAM_LEFT_ID)
right_record = camera_record(REFERENCE_JSON, CAM_RIGHT_ID)
K_LEFT_REFERENCE = scaled_intrinsics(left_record)
K_RIGHT_REFERENCE = scaled_intrinsics(right_record)
D_LEFT_REFERENCE = np.asarray(left_record.get("distortionCoeff", []), dtype=np.float64).reshape(-1, 1)
D_RIGHT_REFERENCE = np.asarray(right_record.get("distortionCoeff", []), dtype=np.float64).reshape(-1, 1)
if D_LEFT_REFERENCE.size == 0 or D_RIGHT_REFERENCE.size == 0:
    raise ValueError("Reference JSON is missing distortion coefficients for CAM_B or CAM_C.")
R_REFERENCE, T_REFERENCE, left_record, right_record = reference_stereo_extrinsics(REFERENCE_JSON)

loaded_pairs = []
for name, left_path, right_path in PAIR_FILES:
    left_image = cv2.imread(str(left_path), cv2.IMREAD_GRAYSCALE)
    right_image = cv2.imread(str(right_path), cv2.IMREAD_GRAYSCALE)
    if left_image is None or right_image is None:
        raise ValueError(f"Could not read image pair: {name}")
    if left_image.shape != (IMAGE_HEIGHT, IMAGE_WIDTH) or right_image.shape != (IMAGE_HEIGHT, IMAGE_WIDTH):
        raise ValueError(
            f"{name}: expected two {IMAGE_WIDTH}x{IMAGE_HEIGHT} images; "
            f"got {left_image.shape[::-1]} and {right_image.shape[::-1]}. Do not resize calibration images."
        )
    loaded_pairs.append((name, left_path, right_path, left_image, right_image))

print("Reference:", REFERENCE_PATH.name)
print("Matched synchronized pairs:", len(loaded_pairs))
print("Image validation: passed")

In [ ]:
OBJECT_TEMPLATE = np.zeros((PATTERN_SIZE[0] * PATTERN_SIZE[1], 3), dtype=np.float32)
OBJECT_TEMPLATE[:, :2] = (
    np.mgrid[0:PATTERN_SIZE[0], 0:PATTERN_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE_MM
)

object_points = []
left_points = []
right_points = []
accepted_names = []
rejected_pairs = []
orientation_log = []
preview_items = []

for name, left_path, right_path, left_image, right_image in loaded_pairs:
    corners_left = detect_board(left_image)
    corners_right = detect_board(right_image)
    if corners_left is None or corners_right is None:
        rejected_pairs.append((name, "Chessboard not detected in both images"))
        continue

    aligned_right, orientation_id, orientation_scores, orientation_margin = align_right_corner_order(
        corners_left, corners_right,
        K_LEFT_REFERENCE, D_LEFT_REFERENCE,
        K_RIGHT_REFERENCE, D_RIGHT_REFERENCE,
        R_REFERENCE, T_REFERENCE
    )
    if not np.all(np.isfinite(aligned_right)):
        rejected_pairs.append((name, "Invalid stereo corner correspondence"))
        continue

    object_points.append(OBJECT_TEMPLATE.copy())
    left_points.append(corners_left.astype(np.float32))
    right_points.append(aligned_right)
    accepted_names.append(name)
    orientation_log.append((name, orientation_id, orientation_margin, min(orientation_scores)))

    if len(preview_items) < 4:
        left_view = cv2.cvtColor(left_image, cv2.COLOR_GRAY2RGB)
        right_view = cv2.cvtColor(right_image, cv2.COLOR_GRAY2RGB)
        cv2.drawChessboardCorners(left_view, PATTERN_SIZE, corners_left, True)
        cv2.drawChessboardCorners(right_view, PATTERN_SIZE, aligned_right, True)
        preview_items.append((name, left_view, right_view))

print(f"Accepted pairs: {len(accepted_names)} / {len(loaded_pairs)}")
print("Rejected pairs:", len(rejected_pairs))
if rejected_pairs:
    print("Rejected examples:", rejected_pairs[:8])
if len(accepted_names) < MIN_PAIRS:
    raise RuntimeError(
        f"Only {len(accepted_names)} valid pairs remain. At least {MIN_PAIRS} are required. "
        "Check the 9x6 inner-corner setting, image pairing, focus and board visibility."
    )

fig, axes = plt.subplots(len(preview_items), 2, figsize=(12, 3.5 * len(preview_items)))
if len(preview_items) == 1:
    axes = np.array([axes])
for row, (name, left_view, right_view) in enumerate(preview_items):
    axes[row, 0].imshow(left_view)
    axes[row, 0].set_title(f"{name} — CAM_B")
    axes[row, 1].imshow(right_view)
    axes[row, 1].set_title(f"{name} — CAM_C")
    axes[row, 0].axis("off")
    axes[row, 1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def per_view_reprojection_errors(objects, images, rvecs, tvecs, camera_matrix, distortion):
    results = []
    for obj, observed, rvec, tvec in zip(objects, images, rvecs, tvecs):
        projected, _ = cv2.projectPoints(obj, rvec, tvec, camera_matrix, distortion)
        delta = projected.reshape(-1, 2) - observed.reshape(-1, 2)
        results.append(float(np.sqrt(np.mean(np.sum(delta ** 2, axis=1)))))
    return np.asarray(results, dtype=np.float64)


def calibrate_mono(objects, images):
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 200, 1e-9)
    return cv2.calibrateCamera(
        objects, images, IMAGE_SIZE, None, None,
        flags=0, criteria=criteria
    )


count = len(object_points)
validation_count = max(4, int(round(count * 0.20)))
validation_rng = np.random.default_rng(RANDOM_SEED)
validation_indices = sorted(
    validation_rng.choice(count, size=validation_count, replace=False).astype(int).tolist()
)
validation_set = set(validation_indices)
training_indices = [index for index in range(count) if index not in validation_set]
if len(training_indices) < MIN_TRAIN_PAIRS:
    raise RuntimeError("Not enough training pairs after creating the independent validation split.")

select = lambda values, indices: [values[index] for index in indices]
train_objects = select(object_points, training_indices)
train_left = select(left_points, training_indices)
train_right = select(right_points, training_indices)

left_rms, K_left, D_left, left_rvecs, left_tvecs = calibrate_mono(train_objects, train_left)
right_rms, K_right, D_right, right_rvecs, right_tvecs = calibrate_mono(train_objects, train_right)

left_view_errors = per_view_reprojection_errors(
    train_objects, train_left, left_rvecs, left_tvecs, K_left, D_left
)
right_view_errors = per_view_reprojection_errors(
    train_objects, train_right, right_rvecs, right_tvecs, K_right, D_right
)
combined_errors = np.sqrt((left_view_errors ** 2 + right_view_errors ** 2) / 2.0)
median_error = float(np.median(combined_errors))
mad_error = float(np.median(np.abs(combined_errors - median_error)))
outlier_limit = max(median_error + 3.5 * 1.4826 * max(mad_error, 0.05), median_error * 2.5, 0.8)
keep_mask = combined_errors <= outlier_limit
minimum_keep = max(MIN_TRAIN_PAIRS, int(math.ceil(0.75 * len(training_indices))))
if int(keep_mask.sum()) < minimum_keep:
    keep_mask[:] = True

if not np.all(keep_mask):
    train_objects = [value for value, keep in zip(train_objects, keep_mask) if keep]
    train_left = [value for value, keep in zip(train_left, keep_mask) if keep]
    train_right = [value for value, keep in zip(train_right, keep_mask) if keep]
    kept_train_indices = [value for value, keep in zip(training_indices, keep_mask) if keep]
else:
    kept_train_indices = list(training_indices)

if len(train_objects) < MIN_TRAIN_PAIRS:
    raise RuntimeError("Too few training pairs remain after outlier screening.")

left_rms, K_left, D_left, left_rvecs, left_tvecs = calibrate_mono(train_objects, train_left)
right_rms, K_right, D_right, right_rvecs, right_tvecs = calibrate_mono(train_objects, train_right)

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 200, 1e-9)
stereo_rms, K_left, D_left, K_right, D_right, R, T, E, F = cv2.stereoCalibrate(
    train_objects,
    train_left,
    train_right,
    K_left,
    D_left,
    K_right,
    D_right,
    IMAGE_SIZE,
    criteria=criteria,
    flags=cv2.CALIB_FIX_INTRINSIC
)

R1, R2, P1, P2, Q, roi_left, roi_right = cv2.stereoRectify(
    K_left, D_left, K_right, D_right, IMAGE_SIZE, R, T,
    flags=cv2.CALIB_ZERO_DISPARITY,
    alpha=0
)

print("Training pairs:", len(train_objects))
print("Validation pairs:", len(validation_indices))
print(f"CAM_B training RMS: {left_rms:.4f} px")
print(f"CAM_C training RMS: {right_rms:.4f} px")
print(f"Stereo training RMS: {stereo_rms:.4f} px")

In [ ]:
def mono_holdout_error(obj, observed, camera_matrix, distortion):
    ok, rvec, tvec = cv2.solvePnP(
        obj.astype(np.float32), observed.astype(np.float32),
        camera_matrix, distortion, flags=cv2.SOLVEPNP_ITERATIVE
    )
    if not ok:
        return float("inf"), None, None
    projected, _ = cv2.projectPoints(obj, rvec, tvec, camera_matrix, distortion)
    delta = projected.reshape(-1, 2) - observed.reshape(-1, 2)
    error = float(np.sqrt(np.mean(np.sum(delta ** 2, axis=1))))
    return error, rvec, tvec


def stereo_transfer_error(obj, observed_target, rvec_source, tvec_source,
                          R_stereo, T_stereo, K_target, D_target):
    R_board_source, _ = cv2.Rodrigues(rvec_source)
    R_board_target = R_stereo @ R_board_source
    rvec_target, _ = cv2.Rodrigues(R_board_target)
    tvec_target = R_stereo @ tvec_source + T_stereo
    projected, _ = cv2.projectPoints(obj, rvec_target, tvec_target, K_target, D_target)
    delta = projected.reshape(-1, 2) - observed_target.reshape(-1, 2)
    return float(np.sqrt(np.mean(np.sum(delta ** 2, axis=1))))


left_validation = []
right_validation = []
stereo_transfer_lr = []
stereo_transfer_rl = []
epipolar_values = []

for index in validation_indices:
    obj = object_points[index]
    observed_left = left_points[index]
    observed_right = right_points[index]

    left_error, left_rvec, left_tvec = mono_holdout_error(obj, observed_left, K_left, D_left)
    right_error, right_rvec, right_tvec = mono_holdout_error(obj, observed_right, K_right, D_right)
    left_validation.append(left_error)
    right_validation.append(right_error)

    if left_rvec is not None:
        stereo_transfer_lr.append(
            stereo_transfer_error(obj, observed_right, left_rvec, left_tvec,
                                  R, T, K_right, D_right)
        )
    if right_rvec is not None:
        R_reverse = R.T
        T_reverse = -R_reverse @ T
        stereo_transfer_rl.append(
            stereo_transfer_error(obj, observed_left, right_rvec, right_tvec,
                                  R_reverse, T_reverse, K_left, D_left)
        )

    rectified_left = cv2.undistortPoints(observed_left, K_left, D_left, R=R1, P=P1).reshape(-1, 2)
    rectified_right = cv2.undistortPoints(observed_right, K_right, D_right, R=R2, P=P2).reshape(-1, 2)
    epipolar_values.extend((rectified_left[:, 1] - rectified_right[:, 1]).tolist())

val_left = float(np.mean(left_validation)) if left_validation else float("inf")
val_right = float(np.mean(right_validation)) if right_validation else float("inf")
transfer_left_right = float(np.mean(stereo_transfer_lr)) if stereo_transfer_lr else float("inf")
transfer_right_left = float(np.mean(stereo_transfer_rl)) if stereo_transfer_rl else float("inf")
stereo_transfer_validation = float(np.mean([transfer_left_right, transfer_right_left]))
epipolar_rms = float(np.sqrt(np.mean(np.square(epipolar_values)))) if epipolar_values else float("inf")

reference_translation = left_record.get("extrinsics", {}).get("translation", {})
if int(left_record.get("extrinsics", {}).get("toCameraSocket", -1)) != CAM_RIGHT_ID:
    right_ext = right_record.get("extrinsics", {})
    reference_translation = right_ext.get("translation", {})
    reference_rotation = np.asarray(right_ext.get("rotationMatrix"), dtype=np.float64)
    reference_t = np.array([reference_translation.get(axis, 0.0) for axis in ("x", "y", "z")], dtype=np.float64)
    reference_baseline_cm = float(np.linalg.norm(reference_t))
else:
    reference_t = np.array([reference_translation.get(axis, 0.0) for axis in ("x", "y", "z")], dtype=np.float64)
    reference_baseline_cm = float(np.linalg.norm(reference_t))

new_baseline_cm = float(np.linalg.norm(T) / 10.0)
baseline_change_percent = (
    100.0 * abs(new_baseline_cm - reference_baseline_cm) / reference_baseline_cm
    if reference_baseline_cm > 1e-9 else float("inf")
)

all_corner_points = np.concatenate(
    [point.reshape(-1, 2) for point in left_points + right_points], axis=0
)
coverage_x = float((all_corner_points[:, 0].max() - all_corner_points[:, 0].min()) / IMAGE_WIDTH)
coverage_y = float((all_corner_points[:, 1].max() - all_corner_points[:, 1].min()) / IMAGE_HEIGHT)
centers = np.array([np.mean(point.reshape(-1, 2), axis=0) for point in left_points])
center_spread_x = float((centers[:, 0].max() - centers[:, 0].min()) / IMAGE_WIDTH)
center_spread_y = float((centers[:, 1].max() - centers[:, 1].min()) / IMAGE_HEIGHT)
coverage_ok = coverage_x >= 0.45 and coverage_y >= 0.35 and center_spread_x >= 0.20 and center_spread_y >= 0.15

finite_parameters = all(np.all(np.isfinite(value)) for value in (K_left, D_left, K_right, D_right, R, T, R1, R2, P1, P2, Q))
valid_intrinsics = (
    K_left[0, 0] > 0 and K_left[1, 1] > 0 and K_right[0, 0] > 0 and K_right[1, 1] > 0
    and 0 <= K_left[0, 2] <= IMAGE_WIDTH and 0 <= K_left[1, 2] <= IMAGE_HEIGHT
    and 0 <= K_right[0, 2] <= IMAGE_WIDTH and 0 <= K_right[1, 2] <= IMAGE_HEIGHT
)

metrics = {
    "Environment": ENVIRONMENT,
    "Matched pairs": len(PAIR_FILES),
    "Accepted pairs": len(accepted_names),
    "Rejected pairs": len(rejected_pairs),
    "Training pairs used": len(train_objects),
    "Validation pairs": len(validation_indices),
    "CAM_B training RMS (px)": round(float(left_rms), 4),
    "CAM_C training RMS (px)": round(float(right_rms), 4),
    "Stereo training RMS (px)": round(float(stereo_rms), 4),
    "CAM_B validation reprojection (px)": round(val_left, 4),
    "CAM_C validation reprojection (px)": round(val_right, 4),
    "Stereo transfer validation (px)": round(stereo_transfer_validation, 4),
    "Validation epipolar RMS (px)": round(epipolar_rms, 4),
    "Baseline change vs reference (%)": round(baseline_change_percent, 2),
    "Corner coverage X (%)": round(coverage_x * 100, 1),
    "Corner coverage Y (%)": round(coverage_y * 100, 1),
    "Board-center spread X (%)": round(center_spread_x * 100, 1),
    "Board-center spread Y (%)": round(center_spread_y * 100, 1)
}

display(pd.DataFrame.from_dict(metrics, orient="index", columns=["Result"]))

quality_checks = {
    "finite calibration parameters": finite_parameters,
    "valid camera intrinsics": valid_intrinsics,
    "capture coverage": coverage_ok,
    "CAM_B training RMS": left_rms <= MAX_MONO_RMS_PX,
    "CAM_C training RMS": right_rms <= MAX_MONO_RMS_PX,
    "Stereo training RMS": stereo_rms <= MAX_STEREO_RMS_PX,
    "CAM_B validation reprojection": val_left <= MAX_VALIDATION_REPROJECTION_PX,
    "CAM_C validation reprojection": val_right <= MAX_VALIDATION_REPROJECTION_PX,
    "Stereo transfer validation": stereo_transfer_validation <= MAX_VALIDATION_REPROJECTION_PX,
    "Validation epipolar RMS": epipolar_rms <= MAX_EPIPOLAR_RMS_PX,
    "Baseline consistency": baseline_change_percent <= MAX_BASELINE_CHANGE_PERCENT
}

print("\nQuality checks:")
for name, passed in quality_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")

quality_ok = all(quality_checks.values())
print("\nValidation:", "ACCEPTED" if quality_ok else "REVIEW REQUIRED")

In [ ]:
map_left_x, map_left_y = cv2.initUndistortRectifyMap(
    K_left, D_left, R1, P1, IMAGE_SIZE, cv2.CV_32FC1
)
map_right_x, map_right_y = cv2.initUndistortRectifyMap(
    K_right, D_right, R2, P2, IMAGE_SIZE, cv2.CV_32FC1
)

preview_index = validation_indices[0]
preview_name = accepted_names[preview_index]
preview_pair = next(item for item in loaded_pairs if item[0] == preview_name)
left_preview = cv2.remap(preview_pair[3], map_left_x, map_left_y, cv2.INTER_LINEAR)
right_preview = cv2.remap(preview_pair[4], map_right_x, map_right_y, cv2.INTER_LINEAR)

left_display = cv2.cvtColor(left_preview, cv2.COLOR_GRAY2RGB)
right_display = cv2.cvtColor(right_preview, cv2.COLOR_GRAY2RGB)
for y in range(40, IMAGE_HEIGHT, 40):
    cv2.line(left_display, (0, y), (IMAGE_WIDTH - 1, y), (255, 0, 0), 1)
    cv2.line(right_display, (0, y), (IMAGE_WIDTH - 1, y), (255, 0, 0), 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(left_display)
axes[0].set_title("Rectified CAM_B")
axes[1].imshow(right_display)
axes[1].set_title("Rectified CAM_C")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

if ENVIRONMENT == "Underwater":
    print(
        "Underwater note: this uses a standard pinhole/Brown distortion model. "
        "Test depth against known distances with the final housing and optical port installed."
    )
if IS_BUILTIN_TEST and ENVIRONMENT == "Underwater":
    print("The built-in images are an Air-style synthetic simulation, not underwater test data.")

In [ ]:
if not quality_ok:
    raise RuntimeError(
        "Calibration was not exported because one or more quality checks failed. "
        "Use more varied synchronized raw images, then rerun the notebook. "
        "Do not lower the checks just to force an output file."
    )

if IS_BUILTIN_TEST:
    print("Built-in test passed the workflow checks. No calibration file was exported because the data are synthetic.")
else:
    def depthai_distortion(distortion):
        values = np.asarray(distortion, dtype=np.float64).reshape(-1)
        if values.size != 5:
            raise ValueError(f"Expected OpenCV's 5-coefficient Brown model, got {values.size} values.")
        return np.pad(values, (0, 9)).astype(float).tolist()

    left_ext = left_record.get("extrinsics", {})
    if int(left_ext.get("toCameraSocket", -1)) == CAM_RIGHT_ID:
        spec_translation = left_ext.get("specTranslation", {})
        spec_translation_vector = [
            float(spec_translation.get("x", 0.0)),
            float(spec_translation.get("y", 0.0)),
            float(spec_translation.get("z", 0.0))
        ]
    else:
        right_ext = right_record.get("extrinsics", {})
        reverse_spec = np.array([
            right_ext.get("specTranslation", {}).get("x", 0.0),
            right_ext.get("specTranslation", {}).get("y", 0.0),
            right_ext.get("specTranslation", {}).get("z", 0.0)
        ], dtype=np.float64)
        reverse_rotation = np.asarray(right_ext.get("rotationMatrix"), dtype=np.float64)
        spec_translation_vector = (-reverse_rotation.T @ reverse_spec).astype(float).tolist()

    calibration_handler = dai.CalibrationHandler(REFERENCE_PATH)
    calibration_handler.setCameraIntrinsics(
        CAM_LEFT, K_left.astype(float).tolist(), IMAGE_WIDTH, IMAGE_HEIGHT
    )
    calibration_handler.setCameraIntrinsics(
        CAM_RIGHT, K_right.astype(float).tolist(), IMAGE_WIDTH, IMAGE_HEIGHT
    )
    calibration_handler.setDistortionCoefficients(CAM_LEFT, depthai_distortion(D_left))
    calibration_handler.setDistortionCoefficients(CAM_RIGHT, depthai_distortion(D_right))
    calibration_handler.setCameraExtrinsics(
        CAM_LEFT,
        CAM_RIGHT,
        R.astype(float).tolist(),
        (T.reshape(3) / 10.0).astype(float).tolist(),
        spec_translation_vector
    )
    calibration_handler.setStereoLeft(CAM_LEFT, R1.astype(float).tolist())
    calibration_handler.setStereoRight(CAM_RIGHT, R2.astype(float).tolist())

    output_data = calibration_handler.eepromToJson()
    output_data["stereoUseSpecTranslation"] = False
    with OUTPUT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(output_data, handle, indent=2)

    verified_handler = dai.CalibrationHandler(OUTPUT_PATH)
    verified_json = verified_handler.eepromToJson()
    verified_left = camera_record(verified_json, CAM_LEFT_ID)
    verified_right = camera_record(verified_json, CAM_RIGHT_ID)
    if not np.allclose(np.asarray(verified_left["intrinsicMatrix"], dtype=float), K_left, rtol=1e-5, atol=1e-4):
        raise RuntimeError("The exported CAM_B intrinsics did not pass the read-back check.")
    if not np.allclose(np.asarray(verified_right["intrinsicMatrix"], dtype=float), K_right, rtol=1e-5, atol=1e-4):
        OUTPUT_PATH.unlink(missing_ok=True)
        raise RuntimeError("The exported CAM_C intrinsics did not pass the read-back check.")

    output_ext = verified_left.get("extrinsics", {})
    output_rotation = np.asarray(output_ext.get("rotationMatrix"), dtype=np.float64)
    output_translation = np.array([
        output_ext.get("translation", {}).get(axis, 0.0) for axis in ("x", "y", "z")
    ], dtype=np.float64)
    expected_translation = (T.reshape(3) / 10.0)
    if int(output_ext.get("toCameraSocket", -1)) != CAM_RIGHT_ID:
        OUTPUT_PATH.unlink(missing_ok=True)
        raise RuntimeError("The exported CAM_B-to-CAM_C extrinsics did not pass the read-back check.")
    if not np.allclose(output_rotation, R, rtol=1e-5, atol=1e-4):
        OUTPUT_PATH.unlink(missing_ok=True)
        raise RuntimeError("The exported stereo rotation did not pass the read-back check.")
    if not np.allclose(output_translation, expected_translation, rtol=1e-5, atol=1e-4):
        OUTPUT_PATH.unlink(missing_ok=True)
        raise RuntimeError("The exported stereo translation did not pass the read-back check.")
    verified_left_distortion = np.asarray(verified_left.get("distortionCoeff", []), dtype=np.float64)
    verified_right_distortion = np.asarray(verified_right.get("distortionCoeff", []), dtype=np.float64)
    expected_left_distortion = np.pad(np.asarray(D_left, dtype=np.float64).reshape(-1), (0, 9))
    expected_right_distortion = np.pad(np.asarray(D_right, dtype=np.float64).reshape(-1), (0, 9))
    if (verified_left_distortion.size != 14 or verified_right_distortion.size != 14
            or not np.allclose(verified_left_distortion, expected_left_distortion, rtol=1e-5, atol=1e-7)
            or not np.allclose(verified_right_distortion, expected_right_distortion, rtol=1e-5, atol=1e-7)):
        OUTPUT_PATH.unlink(missing_ok=True)
        raise RuntimeError("The exported distortion values did not pass the DepthAI read-back check.")

    verified_R1 = np.asarray(verified_json.get("stereoRectificationData", {}).get("rectifiedRotationLeft", []), dtype=np.float64)
    verified_R2 = np.asarray(verified_json.get("stereoRectificationData", {}).get("rectifiedRotationRight", []), dtype=np.float64)
    if (verified_R1.shape != (3, 3) or verified_R2.shape != (3, 3)
            or not np.allclose(verified_R1, R1, rtol=1e-5, atol=1e-4)
            or not np.allclose(verified_R2, R2, rtol=1e-5, atol=1e-4)):
        OUTPUT_PATH.unlink(missing_ok=True)
        raise RuntimeError("The exported stereo rectification rotations did not pass the read-back check.")

    print("Calibration JSON validated and saved:", OUTPUT_PATH.name)
    print("Stereo baseline:", f"{new_baseline_cm:.3f} cm")
    print("Factory/reference calibration preserved as the source template.")
    print("No EEPROM write or device flashing was performed.")
    files.download(str(OUTPUT_PATH))